# Ch21: Exploration（探索）— Agent Laboratory 模式

参考 [AgentLaboratory](https://github.com/SamuelSchmidgall/AgentLaboratory) 项目，
演示「多评审 Agent」模式：3 个不同角色的评审 Agent 对研究报告进行多角度评审。

## 核心思想
- 模拟学术论文评审流程：多个 reviewer 独立评审，汇总意见
- 每个 reviewer 有不同的评审风格和关注点
- 最终汇总所有评审意见，给出综合评价

## 与其他章节的关系
- Ch4 Reflection：Agent 自评 → 本章是多 Agent 互评
- Ch19 Evaluation：LLM-as-Judge 单维度 → 本章是多角色多维度
- Ch7 Multi-Agent：多 Agent 协作 → 本章是多 Agent 并行评审

## 第1步：导入依赖

In [7]:
import os
import json

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage


In [8]:

os.environ["OPENAI_API_KEY"] = os.getenv("MIMO_API_KEY", "")
os.environ["OPENAI_API_BASE"] = "https://token-plan-cn.xiaomimimo.com/v1"

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0.3,
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)
print("MiMo 模型初始化完成")

MiMo 模型初始化完成


## 第2步：定义评审 Agent

3 个不同风格的评审 Agent：
1. **严格实验评审**：关注实验设计和数据质量
2. **影响力评审**：关注研究的影响力和实用性
3. **创新性评审**：关注研究的新颖性和原创性

In [9]:
# 评审模板

REVIEW_TEMPLATE = """
请以{reviewer_persona}的身份，评审以下研究报告。

请严格以 JSON 格式输出评审意见：
{{
  "summary": "研究摘要",
  "strengths": ["优点1", "优点2"],
  "weaknesses": ["缺点1", "缺点2"],
  "originality": "1-4分（低/中/高/非常高）",
  "quality": "1-4分",
  "clarity": "1-4分",
  "significance": "1-4分",
  "overall": "1-10分",
  "decision": "Accept 或 Reject",
  "comments": "详细评审意见"
}}

---
研究报告：
{report}
---
"""

# 三个评审 Agent 的角色定义
REVIEWERS = [
    {
        "name": "Reviewer #1（严格实验评审）",
        "persona": "一位严格但公正的评审专家，注重实验设计的严谨性和数据质量。你会仔细检查实验方法是否合理、数据是否可靠、结论是否有充分的证据支持。"
    },
    {
        "name": "Reviewer #2（影响力评审）",
        "persona": "一位严格且批判性强但公正的评审专家，关注研究的影响力和实用性。你会评估研究是否能推动领域发展、是否有实际应用价值、是否能解决重要问题。"
    },
    {
        "name": "Reviewer #3（创新性评审）",
        "persona": "一位严格但思想开放的评审专家，追求新颖的创意和方法。你会评估研究是否提出了新观点、是否使用了新方法、是否有突破性的发现。"
    },
]

print(f"定义了 {len(REVIEWERS)} 个评审 Agent：")
for r in REVIEWERS:
    print(f"  - {r['name']}")

定义了 3 个评审 Agent：
  - Reviewer #1（严格实验评审）
  - Reviewer #2（影响力评审）
  - Reviewer #3（创新性评审）


## 第3步：定义评审函数

每个评审 Agent 独立评审研究报告，输出结构化 JSON。

In [10]:
def review_report(report: str, reviewer: dict) -> dict:
    """单个评审 Agent 评审研究报告"""
    prompt = REVIEW_TEMPLATE.format(
        reviewer_persona=reviewer["persona"],
        report=report
    )

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        text = response.content.strip()
        # 提取 JSON
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].split("```")[0].strip()
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  [{reviewer['name']}] JSON 解析失败: {e}")
        return {"error": str(e), "raw_output": response.content}
    except Exception as e:
        print(f"  [{reviewer['name']}] 评审出错: {e}")
        return {"error": str(e)}

print("评审函数定义完成")

评审函数定义完成


## 第4步：执行多 Agent 评审

用一篇示例研究报告测试多 Agent 评审流程。

In [11]:
# 示例研究报告

sample_report = """
# 基于 RAG 的企业知识问答系统研究

## 摘要
本文提出了一种基于检索增强生成（RAG）的企业知识问答系统架构。
该系统通过结合向量检索和大语言模型，实现了对企业内部文档的高效问答。
实验表明，该系统在准确率上比纯 LLM 方案提升了 35%，比传统搜索方案提升了 22%。

## 方法
1. 文档预处理：将企业文档分块（chunk size=512, overlap=50），使用 BGE 模型向量化
2. 检索阶段：使用 FAISS 进行向量相似度检索，Top-K=5
3. 重排序：使用 Cross-Encoder 对检索结果重排序
4. 生成阶段：将重排序后的文档注入 Prompt，由 LLM 生成答案

## 实验结果
- 数据集：10,000 篇企业内部文档，500 个测试问答对
- 准确率：RAG 系统 87.3%，纯 LLM 52.1%，传统搜索 65.2%
- 延迟：平均 1.2 秒（含检索和生成）
- 成本：每次查询约 0.003 美元

## 结论
RAG 技术可以显著提升企业知识问答的准确率，同时保持较低的成本。
未来工作包括：支持多模态数据、引入 Agent 自主决策检索策略、优化长文档处理。
"""

print("示例研究报告准备完成，开始多 Agent 评审...")
print(f"报告长度: {len(sample_report)} 字符")

示例研究报告准备完成，开始多 Agent 评审...
报告长度: 521 字符


In [12]:
# 执行多 Agent 评审

reviews = []
for i, reviewer in enumerate(REVIEWERS):
    print(f"\n{'='*60}")
    print(f"正在执行 {reviewer['name']} 的评审...")
    print(f"{'='*60}")

    review = review_report(sample_report, reviewer)
    reviews.append({
        "reviewer": reviewer["name"],
        "review": review
    })

    # 打印评审结果
    if "error" not in review:
        print(f"  决定: {review.get('decision', 'N/A')}")
        print(f"  总分: {review.get('overall', 'N/A')}/10")
        print(f"  原创性: {review.get('originality', 'N/A')}/4")
        print(f"  质量: {review.get('quality', 'N/A')}/4")
        print(f"  清晰度: {review.get('clarity', 'N/A')}/4")
        print(f"  重要性: {review.get('significance', 'N/A')}/4")
        strengths = review.get('strengths', [])
        weaknesses = review.get('weaknesses', [])
        if strengths:
            print(f"  优点: {strengths[0] if strengths else '无'}")
        if weaknesses:
            print(f"  缺点: {weaknesses[0] if weaknesses else '无'}")
    else:
        print(f"  评审出错: {review.get('error', '未知错误')}")


正在执行 Reviewer #1（严格实验评审） 的评审...
  决定: Reject
  总分: 6/10
  原创性: 中/4
  质量: 中/4
  清晰度: 中/4
  重要性: 中/4
  优点: 研究问题具有明确的现实意义，针对企业知识管理场景，应用导向清晰。
  缺点: 实验设计严谨性严重不足：未说明数据集（10,000篇文档、500个问答对）的构建、标注过程及质量控制方法；未报告数据划分方式（如训练/验证/测试集划分）；评估指标单一，仅报告准确率，缺乏召回率、F1值、人工评估等更全面的指标。

正在执行 Reviewer #2（影响力评审） 的评审...
  决定: Accept
  总分: 6/10
  原创性: 中/4
  质量: 中/4
  清晰度: 高/4
  重要性: 中/4
  优点: 研究问题具有明确的现实意义和应用价值，直击企业知识管理效率低下的痛点。
  缺点: 研究的原创性有限。所提出的架构是现有RAG技术栈（分块、向量检索、重排序、生成）的直接组合与应用，缺乏针对企业场景的、具有理论或方法论层面的创新性改进。

正在执行 Reviewer #3（创新性评审） 的评审...
  决定: Reject
  总分: 5分/10
  原创性: 2分（中）/4
  质量: 3分（高）/4
  清晰度: 3分（高）/4
  重要性: 2分（中）/4
  优点: 系统架构描述清晰、完整，采用了当前主流的RAG技术栈（如BGE、FAISS、Cross-Encoder），具有较好的工程实践参考价值。
  缺点: 创新性严重不足。研究中使用的所有组件（分块策略、向量模型、检索与重排方法）均为现有成熟技术，未提出任何新的模型、算法或架构改进，本质上是现有技术的工程集成与应用。


## 第5步：汇总评审意见

将所有评审 Agent 的意见汇总，生成综合评审报告。

In [13]:
# 汇总评审意见

print("\n" + "=" * 60)
print("综合评审报告")
print("=" * 60)

accept_count = 0
reject_count = 0
total_score = 0
score_count = 0

for item in reviews:
    reviewer = item["reviewer"]
    review = item["review"]

    print(f"\n--- {reviewer} ---")

    if "error" in review:
        print(f"  评审出错: {review['error']}")
        continue

    # 统计
    decision = review.get("decision", "Reject")
    if decision == "Accept":
        accept_count += 1
    else:
        reject_count += 1

    try:
        score = int(review.get("overall", 0))
        total_score += score
        score_count += 1
    except (ValueError, TypeError):
        pass

    # 打印详细评审
    print(f"  摘要: {review.get('summary', 'N/A')[:80]}...")
    print(f"  决定: {decision}")
    print(f"  总分: {review.get('overall', 'N/A')}/10")

    strengths = review.get('strengths', [])
    weaknesses = review.get('weaknesses', [])
    if strengths:
        print(f"  优点:")
        for s in strengths[:2]:
            print(f"    - {s}")
    if weaknesses:
        print(f"  缺点:")
        for w in weaknesses[:2]:
            print(f"    - {w}")

# 最终结论
print("\n" + "=" * 60)
print("最终结论")
print("=" * 60)
print(f"Accept: {accept_count} 票, Reject: {reject_count} 票")
if score_count > 0:
    avg_score = total_score / score_count
    print(f"平均分: {avg_score:.1f}/10")
if accept_count > reject_count:
    print("\n最终决定: Accept ✅")
else:
    print("\n最终决定: Reject ❌")


综合评审报告

--- Reviewer #1（严格实验评审） ---
  摘要: 本研究提出了一种基于检索增强生成（RAG）的企业知识问答系统架构，通过结合向量检索（BGE模型、FAISS）与大语言模型，旨在提升对企业内部文档的问答准确率。实...
  决定: Reject
  总分: 6/10
  优点:
    - 研究问题具有明确的现实意义，针对企业知识管理场景，应用导向清晰。
    - 方法描述相对清晰，流程步骤（预处理、检索、重排序、生成）完整，关键参数（如chunk size, overlap, Top-K）有明确说明。
  缺点:
    - 实验设计严谨性严重不足：未说明数据集（10,000篇文档、500个问答对）的构建、标注过程及质量控制方法；未报告数据划分方式（如训练/验证/测试集划分）；评估指标单一，仅报告准确率，缺乏召回率、F1值、人工评估等更全面的指标。
    - 基准方案设置可能不公平：纯LLM方案未使用任何检索增强，与RAG系统比较的公平性存疑；传统搜索方案未说明具体实现（如BM25），其“准确率”的定义和计算方式不明确。

--- Reviewer #2（影响力评审） ---
  摘要: 本文提出了一种基于检索增强生成（RAG）的企业知识问答系统架构，通过结合向量检索（FAISS）和大语言模型（LLM），并引入Cross-Encoder重排序，旨...
  决定: Accept
  总分: 6/10
  优点:
    - 研究问题具有明确的现实意义和应用价值，直击企业知识管理效率低下的痛点。
    - 系统架构设计完整，涵盖了从文档预处理、检索、重排序到生成的全流程，技术选型（如BGE、FAISS、Cross-Encoder）符合当前主流实践，具备较强的工程参考价值。
  缺点:
    - 研究的原创性有限。所提出的架构是现有RAG技术栈（分块、向量检索、重排序、生成）的直接组合与应用，缺乏针对企业场景的、具有理论或方法论层面的创新性改进。
    - 实验评估维度较为单一。主要依赖准确率，缺乏对答案相关性、忠实度、幻觉率等更细粒度质量的评估。数据集（500个问答对）的规模和多样性未充分说明，可能影响结论的普适性。

--- Reviewer #3（创新性评审） ---
  摘要: 本文提出并实现了一个基于检索增强生成（

## 总结

### Agent Laboratory 模式

| 组件 | 说明 |
|------|------|
| ReviewersAgent | 多个不同角色的评审 Agent |
| 评审模板 | 结构化 JSON 输出（15+ 维度） |
| 汇总逻辑 | 投票 + 平均分 + 优缺点汇总 |

### 核心要点
- **多角色评审**比单一评审更全面，减少偏见
- **结构化输出**（JSON）便于自动化汇总和分析
- **投票机制**（Accept/Reject）模拟真实学术评审流程

### 与其他模式的关系

| 模式 | 评审方式 | 适用场景 |
|------|---------|----------|
| Ch4 Reflection | Agent 自评 | 个人写作优化 |
| Ch19 LLM-as-Judge | 单 Agent 多维度 | 质量监控 |
| Ch21 Agent Lab | 多 Agent 多角色 | 学术评审、方案评估 |

### 实际应用
- 学术论文自动评审
- 代码 Review（不同角色审查安全性/性能/可读性）
- 方案评估（不同利益相关者视角）
- 产品需求评审（PM/开发/测试不同视角）